In [0]:
CREATE or REFRESH STREAMING TABLE bronze_customers
COMMENT 'This is customer bronze table'
TBLPROPERTIES('quality' = 'bronze')
AS
select *, _metadata.file_path as filepath, current_timestamp() as ingestion_ts
from cloud_Files (
  '/Volumes/circuit_box/landing/circuit_volume/customers/',
  'json', 
  map('cloudFiles.inferColumnTypes', 'true')
);

In [0]:
CREATE or REFRESH STREAMING TABLE silver_customers_clean(
CONSTRAINT valid_customer_id EXPECT (customer_id is not null) ON VIOLATION Fail Update,
CONSTRAINT valid_customer_name_id EXPECT (customer_name is not null) ON VIOLATION Drop Row,
CONSTRAINT valid_length_phone EXPECT (length(telephone) >= 10),
CONSTRAINT valid_email EXPECT (email is not null), 
CONSTRAINT valid_date EXPECT(date_of_birth> '1920-01-01')
)
COMMENT 'This is customer silver clean table'
TBLPROPERTIES('quality' = 'silver')
AS
select customer_id, customer_name, CAST(date_of_birth as date), email, telephone, CAST(created_date as date) from STREAM(LIVE.bronze_customers)


In [0]:
CREATE or REFRESH STREAMING TABLE silver_customers

In [0]:
APPLY changes into silver_customers
from STREAM(Live.silver_customers_clean)
keys (customer_id)
sequence by created_date
STORED as SCD Type 1